In [ ]:
import pandas as pd

print("pandas", pd.__version__)

pandas 2.2.3


In [ ]:
marks = pd.DataFrame({
    "matric":    ["S101", "S102", "S103", "S104", "S105", "S106", "S107", "S108"],
    "name":      ["Aina", "Ben", "Chong", "Devi", "Emir", "Farah", "Gopal", "Hana"],
    "programme": ["DS", "IT", "DS", "SE", "IT", "DS", "SE", "CS"],
    "quiz":      [16, 12, 18, None, 9, 8, 16, 13],  # out of 20 (Devi missed the quiz)
    "lab":       [24, 20, 28, 22, 16, 14, 25, 26],  # out of 30
})

# ②
print(marks.head(3))

# ③ An attribute, so no brackets
print(marks.shape)

# ④ Also an attribute
print(marks.dtypes)

  matric   name programme  quiz  lab
0   S101   Aina        DS  16.0   24
1   S102    Ben        IT  12.0   20
2   S103  Chong        DS  18.0   28
(8, 5)
matric        object
name          object
programme     object
quiz         float64
lab            int64
dtype: object


In [ ]:
# Check: run this after the cell above
assert isinstance(marks, pd.DataFrame), "① marks should be a DataFrame"
assert marks.shape == (8, 5), "① marks should have 8 rows and 5 columns"
assert list(marks.columns) == ["matric", "name", "programme", "quiz", "lab"], "① keep the column names as given"
print("✅ Task 1 done")

✅ Task 1 done


In [ ]:
# ① The question is asked of every row, so it becomes Boolean masks
is_ds = marks["programme"] == "DS"
good_lab = marks["lab"] >= 24

# ① + ② Keep the rows where both masks are True, and only two columns
top_ds = marks.loc[is_ds & good_lab, ["name", "lab"]]

# ③
top_ds = top_ds.sort_values("lab", ascending=False)

# ④
print(top_ds)

    name  lab
2  Chong   28
0   Aina   24


In [ ]:
# Check: run this after the cell above
assert list(top_ds.columns) == ["name", "lab"], "② keep only the name and lab columns, in that order"
assert top_ds["name"].tolist() == ["Chong", "Aina"], "① or ③ expected Chong, then Aina. Is 24 itself allowed? Which way is the sort?"
print("✅ Task 2 done")

✅ Task 2 done


In [ ]:
# ①
n_missing = marks["quiz"].isna().sum()

# ② One question about the whole table, so an ordinary if
if n_missing > 0:
    # ③ One column name gives a Series of names
    missing_names = marks.loc[marks["quiz"].isna(), "name"]
    print("No quiz mark:", missing_names.tolist())
else:
    print("No missing quiz marks")

# ④ .mean() and .count() skip missing values
mean_quiz = marks["quiz"].mean()
n_recorded = marks["quiz"].count()

# ⑤
print(f"Mean quiz mark: {mean_quiz:.2f} (from {n_recorded} of {len(marks)} students)")

No quiz mark: ['Devi']
Mean quiz mark: 13.14 (from 7 of 8 students)


In [ ]:
# Check: run this after the cell above
assert n_missing == 1, "① n_missing should be 1: mark each missing value, then add up the True values"
assert missing_names.tolist() == ["Devi"], "③ select the name column for the rows where quiz is missing"
assert n_recorded == 7, "④ count only the recorded quiz marks"
assert round(mean_quiz, 2) == 13.14, "④ take the mean of the quiz column"
print("✅ Task 3 done")

✅ Task 3 done


In [ ]:
# ① ② Column arithmetic runs once for every row: no loop needed
marks["coursework"] = marks["quiz"] + marks["lab"]
marks["coursework_pct"] = marks["coursework"] / 50 * 100

# ③ A Yes/No decision whose answers are True/False is just a comparison
marks["passed"] = marks["coursework_pct"] >= 50

# ④
print(marks[["name", "coursework_pct", "passed"]])

    name  coursework_pct  passed
0   Aina            80.0    True
1    Ben            64.0    True
2  Chong            92.0    True
3   Devi             NaN   False
4   Emir            50.0    True
5  Farah            44.0   False
6  Gopal            82.0    True
7   Hana            78.0    True


In [ ]:
# Check: run this after the cell above
assert marks["coursework_pct"].dropna().round(1).tolist() == [80.0, 64.0, 92.0, 50.0, 44.0, 82.0, 78.0], "①② coursework_pct is (quiz + lab) ÷ 50 × 100"
assert marks["passed"].sum() == 6, "③ passed should be True for 50 or more. Look at Emir"
print("✅ Task 4 done")

✅ Task 4 done


In [ ]:
# ① – ④ A chain of decisions becomes if / elif / else
def grade_band(pct):
    if pd.isna(pct):   # ①
        return "Incomplete"
    elif pct >= 80:  # ②
        return "A"
    elif pct >= 65:         # ③
        return "B"
    elif pct >= 50:    # ④
        return "C"
    else:
        return "F"

# ⑤ Pass the function itself: no brackets after its name
marks["grade"] = marks["coursework_pct"].apply(grade_band)

# ⑥
print(marks["grade"].value_counts())

grade
A             3
C             2
Incomplete    1
F             1
B             1
Name: count, dtype: int64


In [ ]:
# Check: run this after the cell above
assert grade_band(80) == "A" and grade_band(79.9) == "B", "② 80 itself should be an A"
assert grade_band(65) == "B" and grade_band(64.9) == "C", "③ 65 itself should be a B"
assert grade_band(50) == "C" and grade_band(49.9) == "F", "④ 50 itself should be a C"
assert grade_band(float("nan")) == "Incomplete", "① a missing value should be Incomplete"
assert marks["grade"].tolist() == ["A", "C", "A", "Incomplete", "C", "F", "A", "B"], "⑤ apply grade_band to coursework_pct"
print("✅ Task 5 done")

✅ Task 5 done


In [ ]:
programmes = pd.DataFrame({
    "programme": ["DS", "IT", "SE"],
    "programme_name": ["Data Science", "Information Technology", "Software Engineering"],
})

# ① One question about the whole lookup table, so an ordinary if
if programmes["programme"].duplicated().any():
    raise ValueError("Fix the lookup table first: a programme code is repeated")

# ②
enriched = marks.merge(
    programmes,
    on="programme",
    how="left",
    validate="many_to_one",
)

# ③
assert len(enriched) == len(marks), "The join lost or duplicated students"

# ④
unmatched = enriched.loc[enriched["programme_name"].isna(), ["name", "programme"]]

# ⑤
print(unmatched)

   name programme
7  Hana        CS


In [ ]:
# Check: run this after the cell above
assert "programme_name" in enriched.columns and len(enriched) == 8, "② left-join programmes onto marks"
assert unmatched["name"].tolist() == ["Hana"], "④ select the rows whose programme_name is missing"
print("✅ Task 6 done")

✅ Task 6 done


In [ ]:
summary = (
    marks.groupby("programme", as_index=False)   # ①
    .agg(
        n_students=("matric", "count"),          # ②
        n_complete=("coursework_pct", "count"),  # ③
        mean_pct=("coursework_pct", "mean"),    # ④
        n_passed=("passed", "sum"),            # ⑤ True counts as 1
    )
    .sort_values("mean_pct", ascending=False)       # ⑥
)

# ⑦
print(summary.round(1))

  programme  n_students  n_complete  mean_pct  n_passed
3        SE           2           1      82.0         1
0        CS           1           1      78.0         1
1        DS           3           3      72.0         2
2        IT           2           2      57.0         2


In [ ]:
# Check: run this after the cell above
assert list(summary.columns) == ["programme", "n_students", "n_complete", "mean_pct", "n_passed"], "keep the column names given"
assert summary["programme"].tolist() == ["SE", "CS", "DS", "IT"], "⑥ sort by mean_pct, highest first"
assert summary["n_students"].tolist() == [2, 1, 3, 2], "② count every row in the group, missing values included"
assert summary["n_complete"].tolist() == [1, 1, 3, 2], "③ count only the recorded coursework_pct values"
assert summary["n_passed"].tolist() == [1, 1, 2, 2], "⑤ add up the True values"
print("✅ Task 7 done")

✅ Task 7 done


In [ ]:
# ① Keep only the students with a recorded coursework_pct
complete = marks.dropna(subset=["coursework_pct"])

# ② Left-join the programme names onto complete
enriched = complete.merge(programmes, on="programme", how="left", validate="m:1")

# ③ & ④ Check for missing programme names and fill them with "Unknown programme"
if enriched["programme_name"].isna().any():
    enriched["programme_name"] = enriched["programme_name"].fillna("Unknown programme")

# ⑤ For every programme_name: count students and calculate pass rate percentage
pass_rates = (
    enriched.groupby("programme_name", as_index=False)
    .agg(
        n_students=("name", "count"),
        pass_rate=("passed", lambda x: x.mean() * 100)
    )
)

# ⑥ Sort by pass_rate (highest first), then programme_name (A -> Z)
pass_rates = pass_rates.sort_values(
    by=["pass_rate", "programme_name"],
    ascending=[False, True]
)

# ⑦ Print the final table rounded to 1 decimal place
print(pass_rates.round(1))

           programme_name  n_students  pass_rate
1  Information Technology           2      100.0
2    Software Engineering           1      100.0
3       Unknown programme           1      100.0
0            Data Science           3       66.7


In [ ]:
# Check: run this after the cell above
assert list(pass_rates.columns) == ["programme_name", "n_students", "pass_rate"], "name the columns programme_name, n_students and pass_rate"
assert pass_rates["n_students"].sum() == 7, "① keep only the students with a recorded coursework_pct"
assert pass_rates["programme_name"].tolist() == ["Information Technology", "Software Engineering", "Unknown programme", "Data Science"], "④ is Hana labelled? ⑥ sort by pass_rate, then programme_name"
assert pass_rates["pass_rate"].round(1).tolist() == [100.0, 100.0, 100.0, 66.7], "⑤ pass_rate should be a percentage from 0 to 100"
print("✅ Task 8 done: great work!")

✅ Task 8 done: great work!
